# BRCA cell fractions by PAM50 and immune subtype

Load shared plotting settings, method labels, cell-type definitions, and output paths for the two BRCA subtype analyses.

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from scipy.stats import mannwhitneyu
import matplotlib as mpl
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.7,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.edgecolor": "white",
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

methods = ["SABER", "BayesPrism","DWLS"]
npg_colors = ["#3C5488", "#E64B35", "#00A087", "#4DBBD5", "#F39B7F"]
results_dir = "/Path/to/cancer/results/BRCA2"
prediction_paths = {method: os.path.join(results_dir, f"{method}.csv") for method in methods}
pam50_plots_dir = "/Path/to/output/cancer/BRCA/plots_pam50_boxplots"
os.makedirs(pam50_plots_dir, exist_ok=True)
pam50_order = ["Basal", "Her2", "LumA", "LumB", "Normal"]
pam50_palette = dict(zip(pam50_order, npg_colors))

immune_plots_dir = "/Path/to/output/cancer/BRCA/plots_immune_subtype"
os.makedirs(immune_plots_dir, exist_ok=True)
class_order = ["D", "F", "IE", "IE/F"]
class_palette = dict(zip(class_order, npg_colors))
cell_features = {
    "CAF": ["iCAFs", "myCAFs"],
    "iCAF": "iCAFs",
    "myCAF": "myCAFs",
    "Total T": ["T_cells_CD4", "T_cells_CD8"],
    "CD4 T": "T_cells_CD4",
    "CD8 T": "T_cells_CD8",
}
required_prediction_cols = sorted(
    {
        col
        for feature_cols in cell_features.values()
        for col in (feature_cols if isinstance(feature_cols, list) else [feature_cols])
    }
)


## Test cell proportions across PAM50 subtypes

Match TCGA patients to SABER, DWLS, and BayesPrism predictions, use patients as statistical units, compare the highest-mean subtype with the other PAM50 groups, and save one PDF per method.


In [11]:
missing_prediction_files = [path for path in prediction_paths.values() if not os.path.exists(path)]
if missing_prediction_files:
    raise FileNotFoundError(f"Missing required prediction files: {missing_prediction_files}")

subtype_df = pd.read_csv("/Path/to/cancer/data/BRCA2/TCGA_BRCA_subtype.csv")
subtype_df["patient"] = subtype_df["patient"].astype(str).str[:12]
subtype_labels = subtype_df.set_index("patient")["BRCA_Subtype_PAM50"]
subtype_labels = subtype_labels[subtype_labels.isin(pam50_order)]
for method in methods:
    P_pred = pd.read_csv(prediction_paths[method], index_col=0)
    P_pred.index = P_pred.index.str.replace(".", "-", regex=False).str[:12]
    common_samples = subtype_labels.index.intersection(P_pred.index).drop_duplicates()
    pam50_counts = subtype_labels.loc[common_samples].value_counts()
    (fig, axes) = plt.subplots(nrows=2, ncols=4, figsize=(7, 7))
    axes = axes.flatten()
    for ax, ct in zip(
        axes,
        [
            "Tumor_Basal_SC", "Tumor_Her2_SC", "Tumor_LumA_SC", "Tumor_LumB_SC",
            "Tumor_Cycling","T_cells_CD8", "Plasmablasts", "NK_cells", 
        ],
    ):
        plot_data = pd.DataFrame(
            {
                "Proportion": P_pred.loc[common_samples, ct],
                "Subtype": subtype_labels.loc[common_samples],
            }
        ).dropna()
        plot_data["Subtype"] = pd.Categorical(
            plot_data["Subtype"], categories=pam50_order, ordered=True
        )
        sns.boxplot(
            x="Subtype",
            y="Proportion",
            data=plot_data,
            order=pam50_order,
            palette=pam50_palette,
            ax=ax,
            width=0.58,
            linewidth=0.8,
            fliersize=0,
            saturation=0.95,
            boxprops=dict(edgecolor="black", linewidth=0.8),
            whiskerprops=dict(color="black", linewidth=0.8),
            capprops=dict(color="black", linewidth=0.8),
            medianprops=dict(color="black", linewidth=1.0),
        )
        sns.stripplot(
            x="Subtype",
            y="Proportion",
            data=plot_data,
            order=pam50_order,
            ax=ax,
            color="black",
            size=2.0,
            alpha=0.35,
            jitter=0.22,
            linewidth=0,
        )
        ax.set_title(ct, fontsize=9, fontweight="normal", pad=5)
        ax.set_xlabel("")
        ax.set_ylabel("Proportion", fontsize=8)
        max_subtype = plot_data.groupby("Subtype", observed=False)["Proportion"].mean().idxmax()
        data_min = plot_data["Proportion"].min()
        data_max_value = plot_data["Proportion"].max()
        y_range = data_max_value - data_min
        if y_range == 0:
            y_range = 0.05
        h = y_range * 0.05
        line_y = data_max_value + h
        final_y = line_y
        for i, target_subtype in enumerate(pam50_order):
            if target_subtype == max_subtype:
                continue
            group_max = plot_data[plot_data["Subtype"] == max_subtype]["Proportion"]
            group_target = plot_data[plot_data["Subtype"] == target_subtype]["Proportion"]
            assert len(group_max) > 2 and len(group_target) > 2
            (_, pval) = mannwhitneyu(group_max, group_target, alternative="two-sided")
            if pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = None
            if sig:
                x1 = pam50_order.index(max_subtype)
                x2 = i
                ax.plot(
                    [x1, x1, x2, x2],
                    [line_y, line_y + h / 2, line_y + h / 2, line_y],
                    lw=0.8,
                    c="black",
                )
                ax.text(
                    (x1 + x2) / 2,
                    line_y + h / 2,
                    sig,
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    color="black",
                )
                final_y = line_y + h / 2
                line_y += h * 1.6
        ax.set_ylim(data_min - y_range * 0.08, final_y + y_range * 0.2)
        ax.tick_params(axis="x", which="both", labelbottom=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.7)
        ax.spines["bottom"].set_linewidth(0.7)
        ax.tick_params(axis="both", which="major", width=0.7, length=3, pad=2)
        ax.grid(False)
    fig.suptitle(
        f"{method} - Cell Proportions by BRCA PAM50 Subtype",
        fontsize=13,
        fontweight="normal",
        y=0.98,
    )
    fig.legend(
        handles=[
            Patch(
                facecolor=pam50_palette[subtype],
                edgecolor="black",
                linewidth=0.5,
                label=f"{subtype} (n={pam50_counts.get(subtype, 0)})",
            )
            for subtype in pam50_order
        ],
        loc="lower center",
        ncol=len(pam50_order),
        frameon=False,
        bbox_to_anchor=(0.5, 0.01),
        handlelength=1.1,
        handleheight=0.8,
        handletextpad=0.5,
        columnspacing=1.4,
        fontsize=7,
    )
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig(f"{pam50_plots_dir}/{method}_PAM50_boxplot.pdf", bbox_inches="tight")
    plt.close()


## Test CAF and T-cell proportions across immune subtypes

Match TCGA patients to immune classes, run Mann–Whitney comparisons for SABER, DWLS, and BayesPrism CAF/T-cell features, and save per-method PDFs plus statistics and sample-count CSV files.


In [7]:
immune_df = pd.read_csv("/Path/to/cancer/data/BRCA2/BRCA_immune_subtypes.csv", sep=";")
immune_df = immune_df[["Patient", "Class"]].dropna().copy()
immune_df["Patient"] = pd.Index(immune_df["Patient"]).astype(str).str.replace(".", "-", regex=False).str[:12]
immune_df = immune_df.drop_duplicates(subset="Patient", keep="first").set_index("Patient")
immune_df = immune_df.loc[immune_df["Class"].isin(class_order)].copy()
all_stats_rows = []
method_summary_rows = []
for method in methods:
    prediction_df = pd.read_csv(prediction_paths[method], index_col=0)
    prediction_df.index = pd.Index(prediction_df.index).astype(str).str.replace(".", "-", regex=False).str[:12]
    prediction_df = prediction_df.loc[~prediction_df.index.duplicated(keep="first")].copy()
    prediction_df[required_prediction_cols] = prediction_df[required_prediction_cols].apply(
        pd.to_numeric, errors="coerce"
    )
    common_samples = prediction_df.index.intersection(immune_df.index).drop_duplicates()
    base_plot_data = pd.DataFrame(
        {"Sample": common_samples, "Class": immune_df.loc[common_samples, "Class"].values}
    ).dropna()
    base_plot_data["Class"] = pd.Categorical(
        base_plot_data["Class"], categories=class_order, ordered=True
    )
    base_plot_data = base_plot_data.sort_values("Class").reset_index(drop=True)
    for class_name in class_order:
        method_summary_rows.append(
            {
                "method": method,
                "method_display": method,
                "class": class_name,
                "n_samples": int((base_plot_data["Class"] == class_name).sum()),
            }
        )
    (fig, axes) = plt.subplots(2, 3, figsize=(4.5, 6.4), sharey=False)
    axes = axes.flatten()
    for ax, (cell_type, feature_cols) in zip(axes, cell_features.items()):
        plot_data = base_plot_data.copy()
        if isinstance(feature_cols, list):
            plot_data["Proportion"] = (
                prediction_df.loc[plot_data["Sample"], feature_cols].sum(axis=1, min_count=1).values
            )
        else:
            plot_data["Proportion"] = prediction_df.loc[plot_data["Sample"], feature_cols].values
        plot_data = plot_data.dropna(subset=["Class", "Proportion"]).copy()
        class_means = (
            plot_data.groupby("Class", observed=False)["Proportion"].mean().reindex(class_order)
        )
        available_class_means = class_means.dropna()
        assert not available_class_means.empty
        reference_class = available_class_means.idxmax()
        reference_values = plot_data.loc[plot_data["Class"] == reference_class, "Proportion"]
        stats_rows = []
        for comparison_class in class_order:
            if comparison_class == reference_class:
                continue
            comparison_values = plot_data.loc[plot_data["Class"] == comparison_class, "Proportion"]
            assert len(reference_values) > 2 and len(comparison_values) > 2
            (u_stat, p_value) = mannwhitneyu(
                reference_values, comparison_values, alternative="two-sided"
            )
            row = {
                "method": method,
                "method_display": method,
                "cell_type": cell_type,
                "reference_class": reference_class,
                "comparison_class": comparison_class,
                "n_reference": len(reference_values),
                "n_comparison": len(comparison_values),
                "mean_reference": reference_values.mean(),
                "mean_comparison": comparison_values.mean(),
                "median_reference": reference_values.median(),
                "median_comparison": comparison_values.median(),
                "u_statistic": u_stat,
                "p_value": p_value,
                "significance": "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns",
            }
            stats_rows.append(row)
            all_stats_rows.append(row)
        sns.boxplot(
            x="Class",
            y="Proportion",
            data=plot_data,
            order=class_order,
            palette=class_palette,
            ax=ax,
            width=0.58,
            linewidth=0.8,
            fliersize=0,
            saturation=0.95,
            boxprops={"edgecolor": "black", "linewidth": 0.8},
            whiskerprops={"color": "black", "linewidth": 0.8},
            capprops={"color": "black", "linewidth": 0.8},
            medianprops={"color": "black", "linewidth": 1.0},
        )
        sns.stripplot(
            x="Class",
            y="Proportion",
            data=plot_data,
            order=class_order,
            ax=ax,
            color="black",
            size=1.8,
            alpha=0.35,
            jitter=0.22,
            linewidth=0,
        )
        ax.set_title(cell_type, fontsize=9, fontweight="normal", pad=5)
        ax.set_xlabel("")
        ax.set_ylabel(f"{method}-predicted proportion")
        data_min = plot_data["Proportion"].min()
        data_max = plot_data["Proportion"].max()
        y_range = data_max - data_min
        if pd.isna(y_range) or y_range == 0:
            y_range = max(abs(data_max) * 0.1, 0.05)
        significant_rows = [row for row in stats_rows if row["significance"] != "ns"]
        h = y_range * 0.05
        step = h * 1.6
        line_y = data_max + h
        final_y = data_max
        x_reference = class_order.index(reference_class)
        significant_rows = sorted(
            significant_rows,
            key=lambda row: abs(class_order.index(row["comparison_class"]) - x_reference),
        )
        for row in significant_rows:
            x_comparison = class_order.index(row["comparison_class"])
            left_x = min(x_reference, x_comparison)
            right_x = max(x_reference, x_comparison)
            ax.plot(
                [left_x, left_x, right_x, right_x],
                [line_y, line_y + h / 2, line_y + h / 2, line_y],
                linewidth=0.8,
                color="black",
                clip_on=False,
            )
            ax.text(
                (left_x + right_x) / 2,
                line_y + h / 2,
                row["significance"],
                ha="center",
                va="bottom",
                fontsize=8,
                color="black",
            )
            final_y = line_y + h / 2
            line_y += step
        lower_limit = data_min - y_range * 0.08
        if significant_rows:
            upper_limit = final_y + y_range * 0.2
        else:
            upper_limit = data_max + y_range * 0.12
        ax.set_ylim(lower_limit, upper_limit)
        ax.tick_params(axis="x", which="both", labelbottom=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.7)
        ax.spines["bottom"].set_linewidth(0.7)
        ax.tick_params(axis="both", which="major", width=0.7, length=3, pad=2)
        ax.grid(False)
    fig.suptitle(
        f"{method}: CAF subtypes and T cells by BRCA immune subtype",
        fontsize=11,
        fontweight="normal",
        y=1.03,
    )
    class_counts = (
        base_plot_data["Class"].value_counts(sort=False).reindex(class_order, fill_value=0)
    )
    fig.legend(
        handles=[
            Patch(
                facecolor=class_palette[class_name],
                edgecolor="black",
                linewidth=0.5,
                label=f"{class_name} (n={int(class_counts[class_name])})",
            )
            for class_name in class_order
        ],
        loc="lower center",
        ncol=len(class_order),
        frameon=False,
        bbox_to_anchor=(0.5, 0.01),
        handlelength=1.1,
        handleheight=0.8,
        handletextpad=0.5,
        columnspacing=1.4,
        fontsize=7,
    )
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    fig.savefig(
        os.path.join(immune_plots_dir, f"{method}_BRCA2_CAFsubtypes_Tcells_by_immune_subtype_boxplot.pdf"),
        bbox_inches="tight",
    )
    plt.close(fig)
pd.DataFrame(
    all_stats_rows,
    columns=[
        "method", "method_display", "cell_type", "reference_class", "comparison_class",
        "n_reference", "n_comparison", "mean_reference", "mean_comparison",
        "median_reference", "median_comparison", "u_statistic", "p_value", "significance",
    ],
).to_csv(
    os.path.join(immune_plots_dir, "All_methods_BRCA2_CAFsubtypes_Tcells_by_immune_subtype_stats.csv"),
    index=False,
)
pd.DataFrame(method_summary_rows).to_csv(
    os.path.join(immune_plots_dir, "All_methods_BRCA2_immune_subtype_sample_counts.csv"), index=False
)
